# Gestión de Errores y Resiliencia en APIs de IA

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/apis/06-gestion-errores-resilience.ipynb)

Retry con backoff, circuit breaker y fallback multiprovedor para APIs de IA.

Las APIs de LLMs fallan: rate limits, timeouts, sobrecarga del proveedor, errores transitorios de red. Este notebook construye una capa de resiliencia completa, desde reintentos inteligentes hasta un cliente que cambia automáticamente de proveedor cuando uno falla.

In [ ]:
%pip install anthropic openai tenacity --quiet

import os
import time
import random
import json
import hashlib
from enum import Enum
from dataclasses import dataclass, field
from typing import Optional, Callable
from datetime import datetime, timedelta

print('Dependencias listas.')

## 1. Clasificación de errores de API

No todos los errores deben reintentarse. La estrategia correcta depende del tipo:

| Código | Tipo | ¿Reintentar? | Estrategia |
|---|---|---|---|
| 400 | Bad Request | No | Corregir el prompt |
| 401 | Unauthorized | No | Verificar API key |
| 429 | Rate Limit | Sí | Esperar y reintentar |
| 500 | Server Error | Sí | Backoff exponencial |
| 529 | Overloaded | Sí | Cambiar proveedor |

In [ ]:
class TipoError(Enum):
    RATE_LIMIT = "rate_limit"
    SERVER_ERROR = "server_error"
    TIMEOUT = "timeout"
    AUTH_ERROR = "auth_error"
    BAD_REQUEST = "bad_request"
    OVERLOADED = "overloaded"

def clasificar_error(status_code: int, mensaje: str = "") -> TipoError:
    """Clasifica un error HTTP para decidir la estrategia de recuperación."""
    if status_code == 429:
        return TipoError.RATE_LIMIT
    elif status_code == 529 or 'overloaded' in mensaje.lower():
        return TipoError.OVERLOADED
    elif status_code >= 500:
        return TipoError.SERVER_ERROR
    elif status_code == 401:
        return TipoError.AUTH_ERROR
    elif status_code == 400:
        return TipoError.BAD_REQUEST
    else:
        return TipoError.TIMEOUT

ERRORES_REINTENTABLES = {TipoError.RATE_LIMIT, TipoError.SERVER_ERROR,
                         TipoError.TIMEOUT, TipoError.OVERLOADED}

# Prueba
for codigo, msg in [(429, ''), (500, ''), (401, ''), (529, 'API overloaded')]:
    tipo = clasificar_error(codigo, msg)
    reintentable = tipo in ERRORES_REINTENTABLES
    print(f'HTTP {codigo}: {tipo.value:15s} | reintentable={reintentable}')

## 2. RetryClient con backoff exponencial y jitter

El **jitter** (ruido aleatorio) evita que múltiples clientes reintenten exactamente al mismo tiempo, lo que agravaría el problema de rate limiting (efecto 'thundering herd').

In [ ]:
import anthropic
from anthropic import RateLimitError, InternalServerError, APIStatusError

class RetryConfig:
    max_intentos: int = 5
    espera_base: float = 1.0    # segundos
    espera_max: float = 60.0    # segundos
    factor: float = 2.0         # multiplicador exponencial
    jitter: float = 0.25        # fracción de ruido aleatorio

class RetryClient:
    """Cliente Anthropic con reintentos automáticos y backoff exponencial."""

    def __init__(self, api_key: Optional[str] = None, config: Optional[RetryConfig] = None):
        self.client = anthropic.Anthropic(api_key=api_key or os.environ.get('ANTHROPIC_API_KEY', 'sk-ant-demo'))
        self.config = config or RetryConfig()
        self.estadisticas = {'exitos': 0, 'reintentos': 0, 'fallos': 0}

    def _calcular_espera(self, intento: int) -> float:
        """Backoff exponencial con jitter completo."""
        espera = min(
            self.config.espera_base * (self.config.factor ** intento),
            self.config.espera_max
        )
        jitter_val = espera * self.config.jitter * random.random()
        return espera + jitter_val

    def completar(self, messages: list, model: str = 'claude-haiku-4-5-20251001',
                  max_tokens: int = 1024, **kwargs) -> str:
        """Llama a la API con reintentos automáticos."""
        ultimo_error = None
        for intento in range(self.config.max_intentos):
            try:
                respuesta = self.client.messages.create(
                    model=model, max_tokens=max_tokens,
                    messages=messages, **kwargs
                )
                self.estadisticas['exitos'] += 1
                return respuesta.content[0].text
            except (RateLimitError, InternalServerError) as e:
                ultimo_error = e
                tipo = clasificar_error(getattr(e, 'status_code', 500))
                if tipo not in ERRORES_REINTENTABLES or intento == self.config.max_intentos - 1:
                    self.estadisticas['fallos'] += 1
                    raise
                espera = self._calcular_espera(intento)
                self.estadisticas['reintentos'] += 1
                print(f'  [Intento {intento+1}] Error {tipo.value}. Reintentando en {espera:.1f}s...')
                time.sleep(espera)
        raise ultimo_error

# Demo de tiempos de espera sin llamada real
cfg = RetryConfig()
cliente_demo = RetryClient.__new__(RetryClient)
cliente_demo.config = cfg
print('Tiempos de espera por intento (con jitter, ejemplo):')
for i in range(5):
    espera = cliente_demo._calcular_espera(i)
    print(f'  Intento {i+1}: ~{espera:.2f}s')

## 3. Circuit Breaker

El **circuit breaker** evita llamar a un servicio que está claramente caído. Tiene tres estados:
- **CERRADO**: funcionamiento normal, monitorea fallos
- **ABIERTO**: demasiados fallos recientes, rechaza llamadas inmediatamente
- **SEMI-ABIERTO**: prueba si el servicio se recuperó con una llamada de prueba

In [ ]:
class EstadoCB(Enum):
    CERRADO = "cerrado"        # normal
    ABIERTO = "abierto"        # falla rápido
    SEMI_ABIERTO = "semi_abierto"  # probando recuperación

class CircuitBreaker:
    """Implementación del patrón Circuit Breaker."""

    def __init__(self, umbral_fallos: int = 5, ventana_segundos: float = 60.0,
                 tiempo_reset: float = 30.0):
        self.umbral_fallos = umbral_fallos
        self.ventana = timedelta(seconds=ventana_segundos)
        self.tiempo_reset = tiempo_reset
        self.estado = EstadoCB.CERRADO
        self.fallos: list[datetime] = []
        self.ultimo_fallo: Optional[datetime] = None

    def _limpiar_fallos_viejos(self):
        ahora = datetime.now()
        self.fallos = [t for t in self.fallos if ahora - t < self.ventana]

    def puede_llamar(self) -> bool:
        if self.estado == EstadoCB.CERRADO:
            return True
        if self.estado == EstadoCB.ABIERTO:
            if self.ultimo_fallo and (datetime.now() - self.ultimo_fallo).seconds >= self.tiempo_reset:
                self.estado = EstadoCB.SEMI_ABIERTO
                print('  [CB] Transición → SEMI_ABIERTO (probando recuperación)')
                return True
            return False
        return True  # SEMI_ABIERTO permite una llamada

    def registrar_exito(self):
        if self.estado == EstadoCB.SEMI_ABIERTO:
            self.estado = EstadoCB.CERRADO
            self.fallos.clear()
            print('  [CB] Transición → CERRADO (servicio recuperado)')

    def registrar_fallo(self):
        self.ultimo_fallo = datetime.now()
        if self.estado == EstadoCB.SEMI_ABIERTO:
            self.estado = EstadoCB.ABIERTO
            print('  [CB] Transición → ABIERTO (sigue fallando en prueba)')
            return
        self.fallos.append(self.ultimo_fallo)
        self._limpiar_fallos_viejos()
        if len(self.fallos) >= self.umbral_fallos:
            self.estado = EstadoCB.ABIERTO
            print(f'  [CB] Transición → ABIERTO ({len(self.fallos)} fallos en ventana)')

# Simulación de fallos
cb = CircuitBreaker(umbral_fallos=3)
print('Simulando 3 fallos consecutivos:')
for i in range(4):
    if cb.puede_llamar():
        print(f'  Llamada {i+1}: permitida (estado={cb.estado.value})')
        cb.registrar_fallo()
    else:
        print(f'  Llamada {i+1}: BLOQUEADA por circuit breaker (estado={cb.estado.value})')

## 4. MultiproviderClient: Claude → OpenAI → Caché

El cliente multiprovedor intenta los proveedores en orden. Si todos fallan, devuelve una respuesta cacheada anterior. Esto garantiza disponibilidad máxima.

In [ ]:
class CacheRespuestas:
    """Caché simple en memoria con TTL."""
    def __init__(self, ttl_minutos: int = 60):
        self._cache: dict = {}
        self.ttl = timedelta(minutes=ttl_minutos)

    def _clave(self, messages: list) -> str:
        contenido = json.dumps(messages, ensure_ascii=False, sort_keys=True)
        return hashlib.md5(contenido.encode()).hexdigest()

    def get(self, messages: list) -> Optional[str]:
        clave = self._clave(messages)
        if clave in self._cache:
            respuesta, timestamp = self._cache[clave]
            if datetime.now() - timestamp < self.ttl:
                return respuesta
        return None

    def set(self, messages: list, respuesta: str):
        self._cache[self._clave(messages)] = (respuesta, datetime.now())


class MultiproviderClient:
    """
    Cliente con fallback automático: Anthropic → OpenAI → caché.
    Cada proveedor tiene su propio circuit breaker.
    """

    def __init__(self):
        self.cache = CacheRespuestas(ttl_minutos=30)
        self.circuit_breakers = {
            'anthropic': CircuitBreaker(umbral_fallos=3),
            'openai': CircuitBreaker(umbral_fallos=3)
        }
        self.metricas = {'anthropic': 0, 'openai': 0, 'cache': 0, 'fallos': 0}

    def _llamar_anthropic(self, messages: list, max_tokens: int) -> str:
        client = anthropic.Anthropic()
        resp = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=max_tokens,
            messages=messages
        )
        return resp.content[0].text

    def _llamar_openai(self, messages: list, max_tokens: int) -> str:
        # Importación opcional para no fallar si no está instalado
        try:
            import openai
            client = openai.OpenAI()
            resp = client.chat.completions.create(
                model='gpt-4o-mini',
                max_tokens=max_tokens,
                messages=messages
            )
            return resp.choices[0].message.content
        except ImportError:
            raise RuntimeError('openai no instalado')

    def completar(self, messages: list, max_tokens: int = 512) -> dict:
        """Intenta proveedores en orden con circuit breaker y caché."""
        # 1. Comprobar caché
        cached = self.cache.get(messages)
        if cached:
            self.metricas['cache'] += 1
            return {'respuesta': cached, 'proveedor': 'cache', 'ok': True}

        # 2. Intentar Anthropic
        proveedores = [
            ('anthropic', self._llamar_anthropic),
            ('openai', self._llamar_openai)
        ]
        for nombre, fn in proveedores:
            cb = self.circuit_breakers[nombre]
            if not cb.puede_llamar():
                print(f'  [{nombre}] Circuit breaker ABIERTO, saltando.')
                continue
            try:
                respuesta = fn(messages, max_tokens)
                cb.registrar_exito()
                self.cache.set(messages, respuesta)
                self.metricas[nombre] += 1
                return {'respuesta': respuesta, 'proveedor': nombre, 'ok': True}
            except Exception as e:
                cb.registrar_fallo()
                print(f'  [{nombre}] Error: {type(e).__name__} — probando siguiente.')

        # 3. Fallback: devolver caché aunque haya expirado
        self.metricas['fallos'] += 1
        return {'respuesta': None, 'proveedor': None, 'ok': False,
                'error': 'Todos los proveedores fallaron'}

print('MultiproviderClient listo. Cargará credenciales de variables de entorno.')
print('Proveedores: Anthropic (claude-haiku-4-5-20251001) → OpenAI (gpt-4o-mini) → caché')

## 5. Truncado inteligente de prompts

Un prompt demasiado largo puede superar el context window o provocar errores 400. El truncado inteligente preserva el sistema, la instrucción y un sufijo de contexto reciente.

In [ ]:
def estimar_tokens(texto: str) -> int:
    """Estimación rápida: ~4 caracteres por token (inglés/español)."""
    return len(texto) // 4

def truncar_historial(messages: list, max_tokens: int = 8000,
                      reservar_respuesta: int = 1024) -> list:
    """
    Trunca el historial de mensajes preservando:
    1. El primer mensaje (system o primera instrucción del usuario)
    2. El mensaje más reciente
    3. Los mensajes intermedios más recientes que quepan
    """
    tokens_disponibles = max_tokens - reservar_respuesta

    if not messages:
        return messages

    # Contar tokens del primer y último mensaje (siempre se incluyen)
    primero = messages[0]
    ultimo = messages[-1]
    tokens_fijos = estimar_tokens(str(primero)) + estimar_tokens(str(ultimo))

    if tokens_fijos >= tokens_disponibles:
        return [primero, ultimo]  # solo los imprescindibles

    # Añadir mensajes intermedios desde el más reciente
    tokens_restantes = tokens_disponibles - tokens_fijos
    intermedios_seleccionados = []
    for msg in reversed(messages[1:-1]):
        t = estimar_tokens(str(msg))
        if t <= tokens_restantes:
            intermedios_seleccionados.insert(0, msg)
            tokens_restantes -= t
        else:
            break  # no cabe más

    resultado = [primero] + intermedios_seleccionados + [ultimo]
    tokens_usados = tokens_disponibles - tokens_restantes
    if len(resultado) < len(messages):
        print(f'  Truncado: {len(messages)} → {len(resultado)} mensajes (~{tokens_usados} tokens)')
    return resultado

# Demo
historial_largo = [
    {'role': 'user', 'content': 'Hola, soy Juan. ' + 'Necesito ayuda. ' * 5},
    {'role': 'assistant', 'content': 'Claro, estoy aquí. ' + 'Puedo ayudarte. ' * 5},
    {'role': 'user', 'content': 'Explícame Python. ' + 'Es importante. ' * 10},
    {'role': 'assistant', 'content': 'Python es... ' + 'muy flexible. ' * 20},
    {'role': 'user', 'content': '¿Cómo manejo errores?'}
]

truncado = truncar_historial(historial_largo, max_tokens=200, reservar_respuesta=50)
print(f'Mensajes originales: {len(historial_largo)}')
print(f'Mensajes tras truncar: {len(truncado)}')

## 6. Test de resiliencia completo

Verificamos que todos los componentes funcionan juntos simulando distintos tipos de fallos.

In [ ]:
import unittest

class TestResiliencia(unittest.TestCase):

    def test_clasificacion_errores(self):
        """Los errores HTTP se clasifican correctamente."""
        self.assertEqual(clasificar_error(429), TipoError.RATE_LIMIT)
        self.assertEqual(clasificar_error(500), TipoError.SERVER_ERROR)
        self.assertEqual(clasificar_error(401), TipoError.AUTH_ERROR)
        self.assertEqual(clasificar_error(529, 'overloaded'), TipoError.OVERLOADED)

    def test_errores_reintentables(self):
        """Solo los errores correctos se reintentan."""
        self.assertIn(clasificar_error(429), ERRORES_REINTENTABLES)
        self.assertNotIn(clasificar_error(401), ERRORES_REINTENTABLES)
        self.assertNotIn(clasificar_error(400), ERRORES_REINTENTABLES)

    def test_circuit_breaker_abre(self):
        """El circuit breaker se abre tras N fallos."""
        cb = CircuitBreaker(umbral_fallos=3)
        self.assertEqual(cb.estado, EstadoCB.CERRADO)
        for _ in range(3):
            cb.registrar_fallo()
        self.assertEqual(cb.estado, EstadoCB.ABIERTO)
        self.assertFalse(cb.puede_llamar())

    def test_cache_respuestas(self):
        """La caché devuelve respuestas almacenadas."""
        cache = CacheRespuestas(ttl_minutos=10)
        msgs = [{'role': 'user', 'content': 'Hola'}]
        self.assertIsNone(cache.get(msgs))  # vacía
        cache.set(msgs, 'Hola, ¿cómo estás?')
        self.assertEqual(cache.get(msgs), 'Hola, ¿cómo estás?')

    def test_truncado_historial(self):
        """El truncado preserva el primer y último mensaje."""
        msgs = [{'role': 'user', 'content': f'Msg {i}'} for i in range(20)]
        truncado = truncar_historial(msgs, max_tokens=100, reservar_respuesta=20)
        self.assertEqual(truncado[0], msgs[0])   # primero siempre
        self.assertEqual(truncado[-1], msgs[-1]) # último siempre
        self.assertLessEqual(len(truncado), len(msgs))

# Ejecutar tests
suite = unittest.TestLoader().loadTestsFromTestCase(TestResiliencia)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## Resumen

| Componente | Problema que resuelve | Parámetros clave |
|---|---|---|
| `RetryClient` | Fallos transitorios (429, 500) | `max_intentos`, `factor`, `jitter` |
| `CircuitBreaker` | Evita llamadas a servicio caído | `umbral_fallos`, `tiempo_reset` |
| `MultiproviderClient` | Dependencia de un solo proveedor | Orden de proveedores |
| `truncar_historial` | Errores 400 por prompt demasiado largo | `max_tokens`, `reservar_respuesta` |
| `CacheRespuestas` | Coste y latencia en consultas repetidas | `ttl_minutos` |

### Próximos pasos
- [Notebook 07 — Streaming Avanzado](07-streaming-avanzado.ipynb)
- [Tutorial Async Python para IA](../../python-para-ia/04-async-python-ia.ipynb)